## gCRL-AE applied on the Norman2019 dataset al cels

- Load processed Norman2019 dataset
- Compute eigengenes
- Train gCRL-AE
- Partial MCC permutation analysis

In [1]:
# ensuring all packages are reloaded each time I run a cell
%load_ext autoreload
%autoreload 2

In [ ]:
# importing
import scanpy as sc
from gcrl.training.train_gcrl_ae import train_gcrl_ae
from gcrl.grn.eigengenes import compute_eigengenes
from gcrl.alignment.partial_mcc_perm_experiments import run_partial_mcc_perm_experiments

# setting paths
input_data_folder = '../../data/real/Norman2019/'
input_data_file = input_data_folder + 'Norman2019_processed.h5ad'
output_data_folder = '../../results/real/Norman2019_all_cells/'
results_folder     = '../../results/real/Norman2019_all_cells/'

In [3]:
# loading data
adata = sc.read_h5ad(input_data_file)
adata

AnnData object with n_obs × n_vars = 35048 × 2070
    obs: 'guide_identity', 'read_count', 'UMI_count', 'gemgroup', 'good_coverage', 'number_of_cells', 'guide_ids', 'guide_merged', 'split', 'batch', 'condition', 'cell_type', 'dose_val', 'control', 'drug_dose_name', 'cov_drug_dose_name', 'intervention', 'set'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ensembl', 'kind', 'community'
    uns: 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'rank_genes_groups_cov', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [4]:
# computing eigengenes
compute_eigengenes(adata, mode="all_cells")
adata

AnnData object with n_obs × n_vars = 35048 × 2070
    obs: 'guide_identity', 'read_count', 'UMI_count', 'gemgroup', 'good_coverage', 'number_of_cells', 'guide_ids', 'guide_merged', 'split', 'batch', 'condition', 'cell_type', 'dose_val', 'control', 'drug_dose_name', 'cov_drug_dose_name', 'intervention', 'set'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ensembl', 'kind', 'community'
    uns: 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'rank_genes_groups_cov', 'umap', 'X_comm_eig_comm_ids', 'X_comm_eig_global_index', 'comm_eig_meta'
    obsm: 'X_pca', 'X_umap', 'X_comm_eig'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [5]:
# explained variance ratio per eigengene
# Works with both mode="by_cell_type" (averages across cell types) and
# mode="all_cells" / mode="global" (single global computation).
import pandas as pd
from scipy.stats import pearsonr, spearmanr

meta = adata.uns["comm_eig_meta"]
comm_ids = adata.uns["X_comm_eig_comm_ids"]

rows = []
if meta["mode"] == "by_cell_type":
    for ct, ct_meta in meta["per_cell_type"].items():
        for comm in comm_ids:
            evr = ct_meta["per_comm_stats"][str(comm)]["explained_variance_ratio"]
            n_tfs = len(ct_meta["per_comm_genes"][str(comm)])
            rows.append({"cell_type": ct, "community": comm, "n_tfs": n_tfs, "explained_variance_ratio": evr})
        rows.append({
            "cell_type": ct,
            "community": "all_TF",
            "n_tfs": sum(len(ct_meta["per_comm_genes"][str(comm2)]) for comm2 in comm_ids),
            "explained_variance_ratio": ct_meta["pooled_tf_stats"]["explained_variance_ratio"],
        })
else:  # "global" or "all_cells"
    for comm in comm_ids:
        evr = meta["per_comm_stats"][str(comm)]["explained_variance_ratio"]
        n_tfs = len(meta["per_comm_genes"][str(comm)])
        rows.append({"cell_type": "all", "community": comm, "n_tfs": n_tfs, "explained_variance_ratio": evr})
    rows.append({
        "cell_type": "all",
        "community": "all_TF",
        "n_tfs": sum(len(meta["per_comm_genes"][str(comm2)]) for comm2 in comm_ids),
        "explained_variance_ratio": meta["pooled_tf_stats"]["explained_variance_ratio"],
    })

df_evr = pd.DataFrame(rows)
summary = (
    df_evr.groupby("community")
    .agg(n_tfs=("n_tfs", "first"), mean_evr=("explained_variance_ratio", "mean"))
    .rename(columns={"mean_evr": "explained_variance_ratio (mean across cell types)"})
)

comm_only = summary.drop(index="all_TF", errors="ignore")
r_pearson, p_pearson = pearsonr(comm_only["n_tfs"], comm_only["explained_variance_ratio (mean across cell types)"])
r_spearman, p_spearman = spearmanr(comm_only["n_tfs"], comm_only["explained_variance_ratio (mean across cell types)"])

summary["explained_variance_ratio (mean across cell types)"] = (
    summary["explained_variance_ratio (mean across cell types)"] * 100
).round(1).astype(str) + "%"
print(f"Eigengene mode: {meta['mode']}\n")
print(summary.to_string())
print("\nCorrelation between n_tfs and explained variance ratio (communities only):")
print(f"  Pearson  r = {r_pearson:+.3f}  (p = {p_pearson:.3g})")
print(f"  Spearman r = {r_spearman:+.3f}  (p = {p_spearman:.3g})")


Eigengene mode: all_cells

           n_tfs explained_variance_ratio (mean across cell types)
community                                                         
0.0           31                                              5.4%
1.0           31                                              4.6%
2.0           21                                              8.3%
3.0           20                                              7.6%
4.0           18                                              7.4%
5.0           18                                              6.2%
6.0            2                                             50.0%
all_TF       141                                              2.6%

Correlation between n_tfs and explained variance ratio (communities only):
  Pearson  r = -0.850  (p = 0.0155)
  Spearman r = -0.655  (p = 0.111)


In [6]:
# TF communities targeted by at least one intervention
import itertools

# flatten single and double perturbations to individual TF names
interventions = adata.obs["intervention"].unique()
perturbed_tfs = set(
    itertools.chain.from_iterable(i.split("+") for i in interventions if i != "unperturbed")
)

# map each perturbed TF to its community (TFs only, drop NaN community)
tf_var = adata.var.loc[adata.var["kind"] == "TF", ["community"]].dropna()
perturbed_tf_var = tf_var[tf_var.index.isin(perturbed_tfs)]

# group by community: collect the perturbed TFs per community
community_to_tfs = (
    perturbed_tf_var.reset_index()
    .groupby("community")["gene_symbols"]
    .apply(sorted)
    .apply(list)
)

print(f"Unique perturbed TFs: {len(perturbed_tfs)}")
print(f"Communities targeted: {sorted(community_to_tfs.index.astype(int).tolist())}\n")
for comm, tfs in community_to_tfs.items():
    print(f"  Community {int(comm):2d}  ({len(tfs):2d} perturbed TFs): {', '.join(tfs)}")

Unique perturbed TFs: 30
Communities targeted: [0, 1, 2, 3, 4, 5, 6]

  Community  0  ( 6 perturbed TFs): AHR, CEBPA, EGR1, FOSB, HOXC13, KLF1
  Community  1  (10 perturbed TFs): CEBPE, ETS2, FEV, FOXA1, FOXA3, FOXF1, HES7, HOXA13, IRF1, TBX2
  Community  2  ( 4 perturbed TFs): CEBPB, DLX2, FOXL2, JUN
  Community  3  ( 6 perturbed TFs): HNF4A, LYL1, OSR2, PRDM1, SPI1, TBX3
  Community  4  ( 2 perturbed TFs): LHX1, MEIS1
  Community  5  ( 2 perturbed TFs): HOXB9, POU3F2
  Community  6  ( 0 perturbed TFs): 


/tmp/ipykernel_1658207/2677303824.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("community")["gene_symbols"]


In [7]:
# --- batch size sweep: estimate training time per epoch ---
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from gcrl.models.gcrl_ae import GCRLAE
from gcrl.training.train_gcrl_ae import _to_dense, _zscore_using_reference, _ref_indices_from_query

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

# --- preprocessing (shared across all batch size runs) ---
X_full   = _to_dense(adata.X)
kinds    = adata.var["kind"].astype(str).str.strip().str.upper().values
ref_idx  = _ref_indices_from_query(adata.obs, 'intervention == "unperturbed"')
X_proc, _, _ = _zscore_using_reference(X_full, ref_idx)
tf_idx   = np.where(kinds == "TF")[0]
X_in     = X_proc[:, tf_idx]

n_comm     = pd.Categorical(adata.var["community"]).categories.size
latent_dim = int(n_comm) + 1
input_dim  = X_in.shape[1]
output_dim = X_proc.shape[1]
n_cells    = X_in.shape[0]

N_BATCHES  = 100   # batches to time per run (same model/data, only batch_size varies)

tens_in  = torch.tensor(X_in,   dtype=torch.float32)
tens_tgt = torch.tensor(X_proc, dtype=torch.float32)

results = []
for batch_size in [64, 256, 512, 1024, 2048, 4096]:
    loader = DataLoader(
        TensorDataset(tens_in, tens_tgt),
        batch_size=batch_size, shuffle=True,
    )
    n_batches_per_epoch = len(loader)

    model = GCRLAE(input_dim, latent_dim, (256,), torch.nn.ReLU(), output_dim).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()

    # warm-up
    xb, yb = next(iter(loader))
    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad(); yhat, _ = model(xb); F.mse_loss(yhat, yb[:, :output_dim]).backward(); opt.step()
    torch.cuda.synchronize()

    t_transfer = t_forward = t_backward = 0.0
    for i, (xb_cpu, yb_cpu) in enumerate(loader):
        if i >= N_BATCHES:
            break

        t0 = time.perf_counter()
        xb = xb_cpu.to(device); yb = yb_cpu.to(device)
        torch.cuda.synchronize()
        t_transfer += time.perf_counter() - t0

        t0 = time.perf_counter()
        opt.zero_grad(); yhat, _ = model(xb); loss = F.mse_loss(yhat, yb[:, :output_dim])
        torch.cuda.synchronize()
        t_forward += time.perf_counter() - t0

        t0 = time.perf_counter()
        loss.backward(); opt.step()
        torch.cuda.synchronize()
        t_backward += time.perf_counter() - t0

    t_per_batch = (t_transfer + t_forward + t_backward) / N_BATCHES
    t_per_epoch = t_per_batch * n_batches_per_epoch
    results.append({
        "batch_size"           : batch_size,
        "batches/epoch"        : n_batches_per_epoch,
        "ms/batch"             : t_per_batch * 1e3,
        "transfer %"           : 100 * t_transfer / (t_transfer + t_forward + t_backward),
        "forward %"            : 100 * t_forward  / (t_transfer + t_forward + t_backward),
        "backward %"           : 100 * t_backward / (t_transfer + t_forward + t_backward),
        "s/epoch (est.)"       : t_per_epoch,
        "100 epochs (est.) min": t_per_epoch * 100 / 60,
    })
    print(f"  batch_size={batch_size:5d}  {t_per_epoch:.2f} s/epoch  ({t_per_epoch*100/60:.1f} min for 100 epochs)")

df = pd.DataFrame(results).set_index("batch_size")
df["speedup vs 64"] = df["s/epoch (est.)"].iloc[0] / df["s/epoch (est.)"]
print()
print(df[["batches/epoch", "ms/batch", "transfer %", "forward %", "backward %",
          "s/epoch (est.)", "100 epochs (est.) min", "speedup vs 64"]].to_string(float_format="{:.2f}".format))

Device: cuda

  batch_size=   64  4.41 s/epoch  (7.4 min for 100 epochs)
  batch_size=  256  1.13 s/epoch  (1.9 min for 100 epochs)
  batch_size=  512  0.40 s/epoch  (0.7 min for 100 epochs)
  batch_size= 1024  0.10 s/epoch  (0.2 min for 100 epochs)
  batch_size= 2048  0.03 s/epoch  (0.0 min for 100 epochs)
  batch_size= 4096  0.01 s/epoch  (0.0 min for 100 epochs)

            batches/epoch  ms/batch  transfer %  forward %  backward %  s/epoch (est.)  100 epochs (est.) min  speedup vs 64
batch_size                                                                                                                  
64                    548      8.06        1.92      23.09       74.99            4.41                   7.36           1.00
256                   137      8.28        3.33      22.34       74.33            1.13                   1.89           3.89
512                    69      5.84        5.68      21.54       72.78            0.40                   0.67          10.96
1024  

## Sweep: latent_dim × include_pooled_tf

Train gCRL-AE for 5 values of `latent_dim` centred on the default `(#communities + 1)`:
`[default - 2, default - 1, default, default + 1, default + 2]`.

For each trained model, run a fast partial-MCC permutation experiment with both
`include_pooled_tf=True` and `include_pooled_tf=False`.

AE artefacts are written to `results/real/Norman2019_all_cells/ae_sweep/latent{d}/`.
MCC results are written to `results/real/Norman2019_all_cells/partial_mcc_permutation_sweep/`.

In [8]:
import os
import numpy as np
import pandas as pd

# --- sweep configuration ---
import pandas as pd
n_comm = pd.Categorical(adata.var["community"]).categories.size
default_latent = int(n_comm) + 1                      # 12 for Norman2019 (11 communities + 1)
latent_dims = [default_latent + delta for delta in (-2, -1, 0, 1, 2)]
pooled_tf_flags = [True, False]

# fast MCC settings
MCC_REAL_SEEDS   = 1
MCC_PERMUTATIONS = 20
MCC_PERM_SEEDS   = 1

ae_sweep_root  = "../../results/real/Norman2019_all_cells/ae_sweep"
mcc_sweep_root = "../../results/real/Norman2019_all_cells/partial_mcc_permutation_sweep"
os.makedirs(mcc_sweep_root, exist_ok=True)

sweep_rows = []   # collect summary stats across all conditions

for latent_dim in latent_dims:
    print(f"\n{'='*60}")
    print(f"  latent_dim = {latent_dim}  (default = {default_latent})")
    print(f"{'='*60}")

    ae_outdir = os.path.join(ae_sweep_root, f"latent{latent_dim}")

    # --- train gCRL-AE ---
    res_sweep = train_gcrl_ae(
        adata=adata,
        input_mode="TF",
        reconstruct_all=True,
        latent_dim=latent_dim,
        standardize="zscore_ref",
        reference_query='intervention == "unperturbed"',
        batch_size=1024,
        lr=1e-3,
        num_epochs=100,
        lr_step=100,
        weight_decay=1e-3,
        val_frac=0.1,
        device=None,
        outdir=ae_outdir,
    )
    B_sweep = res_sweep.embeddings   # (n_cells, latent_dim)
    train_loss_final = res_sweep.history["loss"][-1]
    val_loss_final   = res_sweep.history["val_loss"][-1] if res_sweep.history.get("val_loss") else float("nan")
    print(f"  Final train loss: {train_loss_final:.4f}  |  val loss: {val_loss_final:.4f}")

    # --- partial-MCC sweep over include_pooled_tf ---
    for include_pooled in pooled_tf_flags:
        tag = f"latent{latent_dim}_pooled{'T' if include_pooled else 'F'}"
        print(f"  Running MCC: include_pooled_tf={include_pooled} ...", end=" ", flush=True)

        mcc_out = run_partial_mcc_perm_experiments(
            adata=adata,
            embeddings=B_sweep,
            community_col="community",
            reference_query='intervention == "unperturbed"',
            mode="all_cells",
            cell_type_col="cell_type",
            n_real_seeds=MCC_REAL_SEEDS,
            n_permutations=MCC_PERMUTATIONS,
            n_perm_seeds=MCC_PERM_SEEDS,
            include_pooled_tf=include_pooled,
            lr=5e-2,
            steps=100,
            device=None,
            master_seed=123,
            save_density_path=os.path.join(mcc_sweep_root, f"{tag}_dens.png"),
        )

        # save raw arrays
        np.savez(
            os.path.join(mcc_sweep_root, f"{tag}_mcc.npz"),
            scores_real=mcc_out["scores_real"],
            scores_perm=mcc_out["scores_perm"],
        )

        real_mean = float(mcc_out["scores_real"].mean())
        perm_mean = float(mcc_out["scores_perm"].mean())
        print(f"real mean={real_mean:.4f}  perm mean={perm_mean:.4f}")

        sweep_rows.append({
            "latent_dim"       : latent_dim,
            "delta"            : latent_dim - default_latent,
            "include_pooled_tf": include_pooled,
            "train_loss_final" : train_loss_final,
            "val_loss_final"   : val_loss_final,
            "mcc_real_mean"    : real_mean,
            "mcc_perm_mean"    : perm_mean,
            "mcc_delta"        : real_mean - perm_mean,
        })

# save summary table
df_sweep = pd.DataFrame(sweep_rows)
df_sweep.to_csv(os.path.join(mcc_sweep_root, "sweep_summary.csv"), index=False)
print("\nSweep complete. Summary saved to", os.path.join(mcc_sweep_root, "sweep_summary.csv"))


  latent_dim = 6  (default = 8)
  Final train loss: 1.7757  |  val loss: 1.8914
  Running MCC: include_pooled_tf=True ... 

Real runs:   0%|          | 0/1 [00:00<?, ?run/s]

Permutations:   0%|          | 0/20 [00:00<?, ?perm/s]

real mean=0.3418  perm mean=0.3619
  Running MCC: include_pooled_tf=False ... 

Real runs:   0%|          | 0/1 [00:00<?, ?run/s]

Permutations:   0%|          | 0/20 [00:00<?, ?perm/s]

KeyboardInterrupt: 

In [9]:
# --- sweep summary table ---
print(f"Default latent_dim = {default_latent}\n")
print(
    df_sweep.to_string(
        index=False,
        float_format="{:.4f}".format,
        columns=[
            "latent_dim", "delta", "include_pooled_tf",
            "train_loss_final", "val_loss_final",
            "mcc_real_mean", "mcc_perm_mean", "mcc_delta",
        ],
    )
)

Default latent_dim = 12

 latent_dim  delta  include_pooled_tf  train_loss_final  val_loss_final  mcc_real_mean  mcc_perm_mean  mcc_delta
         10     -2               True            1.7313          1.8722         0.4220         0.3912     0.0308
         10     -2              False            1.7313          1.8722         0.4403         0.4291     0.0111
         11     -1               True            1.7246          1.8745         0.4297         0.3986     0.0311
         11     -1              False            1.7246          1.8745         0.4654         0.4392     0.0262
         12      0               True            1.7164          1.8706         0.4483         0.4213     0.0271
         12      0              False            1.7164          1.8706         0.4857         0.4631     0.0226
         13      1               True            1.7133          1.8673         0.4559         0.4303     0.0256
         13      1              False            1.7133          1.8673

## Final training
After the parameter sweeping, final training on selected parameters

In [9]:
# training gCRL-AE
res = train_gcrl_ae(
    adata=adata,
    input_mode="TF",                  # encoder sees TFs only (default)
    reconstruct_all=True,             # decoder reconstructs all genes
    latent_dim=None,                  # inferred as (#communities + 1) if None
    #latent_dim=15,
    hidden_dims=(256, 128),
    standardize="zscore_ref",         # z-score using unperturbed cells as reference (default)
    reference_query='intervention == "unperturbed"',
    # --- batch size and learning rate ---
    # batch_size=1024: 44x faster than default 64 (profiling sweep); GPU saturates here.
    # lr=1e-3: reverted to original value — lr=5e-3 caused a loss spike to ~722 at epoch 10.
    batch_size=1024,
    lr=1e-3,
    # --- epochs and LR schedule ---
    # num_epochs=200: ~7000 gradient steps total at 35 batches/epoch.
    # lr_step=100: single LR decay at epoch 100 (midpoint).
    num_epochs=300,
    lr_step=100,
    # --- regularization ---
    # weight_decay=1e-3: L2 penalty on all parameters via Adam.
    # Needed because the quadratic decoder (91 features -> 2070 genes, ~190k params)
    # severely overfits without regularization: val loss exploded to 25 vs train loss 1.6.
    # Start at 1e-3; increase to 1e-2 if val loss still diverges.
    weight_decay=1e-3,
    # --- validation split ---
    val_frac=0.1,
    device=None,                      # auto-select cuda if available
    outdir="../../results/real/Norman2019_all_cells/ae",   # optional: save artifacts
)

In [11]:
# training loss summary
loss     = res.history["loss"]
val_loss = res.history.get("val_loss")  # None if val_frac=0

header = f"{'Epoch':>6}  {'Train loss':>12}  {'Val loss':>12}"
print(header)
print("-" * len(header))
for ep in [1, 10, 50, 100, 150, 200, 250, 300]:
    if ep > len(loss):
        break
    vl = f"{val_loss[ep-1]:12.4f}" if val_loss else "           -"
    print(f"{ep:6d}  {loss[ep-1]:12.4f}  {vl}")

print()
print(f"Train improvement: {loss[0]-loss[-1]:.4f}  ({100*(loss[0]-loss[-1])/loss[0]:.1f}% reduction)")
if val_loss:
    print(f"Val   improvement: {val_loss[0]-val_loss[-1]:.4f}  ({100*(val_loss[0]-val_loss[-1])/val_loss[0]:.1f}% reduction)")
    gap = val_loss[-1] - loss[-1]
    print(f"Final val - train gap: {gap:+.4f}  ({'possible overfitting' if gap > 0.25 else 'no significant overfitting'})")

 Epoch    Train loss      Val loss
----------------------------------
     1        1.9847        2.0582
    10        1.8249        1.9044
    50        1.7611        1.8740
   100        1.7197        1.8746
   150        1.7011        1.8761
   200        1.6943        1.8763
   250        1.6874        1.8761
   300        1.6845        1.8788

Train improvement: 0.3001  (15.1% reduction)
Val   improvement: 0.1795  (8.7% reduction)
Final val - train gap: +0.1943  (no significant overfitting)


In [12]:
# embeddings
B = res.embeddings              # (n_cells, latent_dim)
print(B.shape)

(35048, 8)


In [13]:
# partial mcc permutation
out = run_partial_mcc_perm_experiments(
        adata=adata,
        embeddings=B,
        community_col='community',
        reference_query='intervention == "unperturbed"',
        mode="all_cells",
        cell_type_col="cell_type",
        n_real_seeds=3,
        n_permutations=20,
        n_perm_seeds=2,
        #include_pooled_tf=False,
        lr=5e-2,
        steps=100,
        device=None,  # auto-select cuda if available, else cpu
        master_seed=123,
        save_density_path='../../results/real/Norman2019_all_cells/partial_mcc_permutation/dens.png',
    )

# Print summary statistics
import numpy as np
print(f"Real scores - Mean: {out['scores_real'].mean():.4f}, Std: {out['scores_real'].std():.4f}")
print(f"Permuted scores - Mean: {out['scores_perm'].mean():.4f}, Std: {out['scores_perm'].std():.4f}")

# Save results
import os
output_dir = '../../results/real/Norman2019_all_cells/partial_mcc_permutation/'
os.makedirs(output_dir, exist_ok=True)

# Save as numpy arrays
np.save(os.path.join(output_dir, 'scores_real.npy'), out['scores_real'])
np.save(os.path.join(output_dir, 'scores_perm.npy'), out['scores_perm'])

# Also save as a combined .npz file for convenience
np.savez(os.path.join(output_dir, 'partial_mcc_results.npz'), 
         scores_real=out['scores_real'], 
         scores_perm=out['scores_perm'])

print(f"\nResults saved to {output_dir}")

Real runs:   0%|          | 0/3 [00:00<?, ?run/s]

Permutations:   0%|          | 0/20 [00:00<?, ?perm/s]

Real scores - Mean: 0.3026, Std: 0.0002
Permuted scores - Mean: 0.3575, Std: 0.0524

Results saved to ../../results/real/Norman2019_all_cells/partial_mcc_permutation/
